In [15]:
import torch
import numpy as np
import torch.nn as nn
from tqdm import trange
from pathlib import Path
import torch.optim as optim
import matplotlib.pyplot as plt
from torch.utils.data import DataLoader, TensorDataset
from sklearn.preprocessing import StandardScaler, MinMaxScaler
import importlib
from torchinfo import summary
import time
import pandas as pd
import pybamm


import sys
sys.path.append('..')
import plot_settings
plot_settings.apply()
import sim_PyBaMM
importlib.reload(sim_PyBaMM);
import exp_pybamm_ver4
importlib.reload(exp_pybamm_ver4);

print(sim_PyBaMM.__file__)
print(exp_pybamm_ver4.__file__) 

/Users/noord/Desktop/MSc_git_new/MSc_thesis/program/_moving_box/data_pybamm/sim_PyBaMM.py
/Users/noord/Desktop/MSc_git_new/MSc_thesis/program/_moving_box/data_pybamm/exp_pybamm_ver4.py


In [29]:
# --- Simulation config ---
T_HORIZON = 3600.0          # [s]  total simulation horizon
T = 3600           # number of time steps
DT = T_HORIZON / (T)   # fixed for all sims
t_fixed = np.linspace(0, T_HORIZON, T)    # fixed 'physical' grid

In [30]:
# --- Data generation ---

translator = {
    'I': 'Current [A]',
    'V': 'Terminal voltage [V]',
    'cn': 'X-averaged negative particle concentration [mol.m-3]',
    'cp': 'X-averaged positive particle concentration [mol.m-3]',
    'cns': 'X-averaged negative particle surface concentration [mol.m-3]',
    'cps': 'X-averaged positive particle surface concentration [mol.m-3]',
    'ocv': 'Battery open-circuit voltage [V]',
}

translator_inv = {v: k for k, v in translator.items()}

data_cols = ['I', 'V', 'ocv']

def gen_cols(data_cols, translator=translator):
    '''Extracting variable names from translator dict for given data_cols'''
    return [translator[col] for col in data_cols]



def time_query(sol, t_fixed):
    t_end = sol["Time [s]"].entries[-1]
    valid = t_fixed <= t_end  # boolean, length T # Valid mask: where simulation was still running
    t_query = np.clip(t_fixed, 0, t_end)     # Observe only up to t_end, fill rest with NaN or last value
    return t_query, valid

def observe_states(t_query, sol, model, data_cols):
    data = pd.DataFrame(index=t_query)
    cols = gen_cols(data_cols)
    for col in cols:
        data[translator_inv[col]] = sol.observe(model.variables[col])(t_query)
    return data

def get_vals_CC(I0,data_cols=data_cols):
    sol, model, param = sim_PyBaMM.simulate_CC(I0, T, T_HORIZON)   # simulate with t_eval = np.linspace(0, t_horizon, T)
    t_query, valid = time_query(sol, t_fixed)
    data = observe_states(t_query, sol, model, data_cols)
    return data, valid, param, sol, model

def get_vals_pulse(data_cols=data_cols):
    sol, model, t_eval, param = exp_pybamm_ver4.generate_data(T_HORIZON, DT, RNG=None)   # simulate with t_eval = np.linspace(0, t_horizon, T)
    t_query, valid = time_query(sol, t_fixed)
    data = observe_states(t_query, sol, model, data_cols)
    return data, valid, param, sol, model

def get_vals_GRF(data_cols=data_cols):
    sol, model, param = sim_PyBaMM.simulate_GRF(T, T_HORIZON)   # simulate with t_eval = np.linspace(0, t_horizon, T)
    t_query, valid = time_query(sol, t_fixed)
    data = observe_states(t_query, sol, model, data_cols)
    return data, valid, param, sol, model


def gen_data(B, data_cols, currents='CC', I_low=0.25, I_high=1.0, seed=None):
    '''Generate trajectories with shape (B, T, D)'''

    rng = np.random.default_rng(seed)

    dot_cols = [f"{k}_dot" for k in data_cols if k in ['cn','cp']]
    all_cols = data_cols + ['dI'] + dot_cols + ['valid']

    D = len(all_cols)

    X = np.zeros((B, T, D))
    var_index = {k:i for i,k in enumerate(all_cols)}

    for b in trange(B):

        if currents == 'CC':
            data, valid_b, param, sol, model = get_vals_CC(float(rng.uniform(I_low, I_high)))
        elif currents == 'pulse':
            data, valid_b, param, sol, model = get_vals_pulse()
        elif currents == 'GRF':
            data, valid_b, param, sol, model = get_vals_GRF()
        else:
            raise ValueError("Invalid current type")

        # base variables
        for col in data_cols:
            X[b, :, var_index[col]] = data[col].values

        # dI
        if 'I' in data_cols:
            I = data['I'].values
            dI = np.zeros_like(I)
            dI[1:] = np.diff(I)
            X[b, :, var_index['dI']] = dI

        # derivatives
        for col in ['cn','cp']:
            if col in data_cols:
                x = data[col].values
                xdot = np.zeros_like(x)
                xdot[:-1] = np.diff(x) / DT
                xdot[-1] = xdot[-2]
                X[b, :, var_index[f"{col}_dot"]] = xdot

        # valid mask
        X[b, :, var_index['valid']] = valid_b.astype(float)

    return X, var_index

In [36]:
import numpy as np
import pandas as pd

def save_pybamm_dataset(X, var_index, t_grid, path, *, pulse=False, Q_ah=1.0, soc0=1.0):
    """Write generated PyBaMM trajectories to a ';'-separated file in the thesis
    schema. Only PyBaMM-derived columns are filled; the mechanical columns
    (u_par, F, u) are left empty for a later COMSOL merge."""
    cols = ['u_par','C','t','V','I','F','u','Ue','soc','eta','pulse','trajectory']
    iI, iV, iOcv, iValid = (var_index['I'], var_index['V'],
                            var_index['ocv'], var_index['valid'])
    frames = []
    for b in range(X.shape[0]):
        valid = X[b, :, iValid].astype(bool)
        t  = np.asarray(t_grid)[valid]
        I  = X[b, valid, iI]
        V  = X[b, valid, iV]
        Ue = X[b, valid, iOcv]
        # Coulomb-counted SOC (discharge positive): soc = soc0 - integral(I dt)/(Q*3600)
        charge_As = np.concatenate([[0.0], np.cumsum(0.5*(I[1:]+I[:-1])*np.diff(t))])
        soc = soc0 - charge_As / (Q_ah * 3600.0)
        frames.append(pd.DataFrame({
            'u_par': 0, 'C': I / Q_ah, 't': t, 'V': V, 'I': I,
            'F': 0, 'u': 0, 'Ue': Ue, 'soc': soc,
            'eta': Ue - V, 'pulse': pulse, 'trajectory': b,
        }, columns=cols))
    out = pd.concat(frames, ignore_index=True)
    out.to_csv(path, sep=';', index=False, na_rep='')
    return out

In [39]:
X, var_index = gen_data(15, data_cols, currents='CC', I_low=0.25, I_high=1.0, seed=42)
print("Data shape:", X.shape)
print("Variable index:", var_index)

100%|██████████| 15/15 [00:53<00:00,  3.54s/it]

Data shape: (15, 3600, 5)
Variable index: {'I': 0, 'V': 1, 'ocv': 2, 'dI': 3, 'valid': 4}


In [42]:
data = save_pybamm_dataset(X, var_index, t_fixed, 'pybamm_CC.txt', pulse=False)

print(data)


       u_par         C            t         V         I  F  u        Ue  \
0          0  0.830467     0.000000  3.759714  0.830467  0  0  3.851821   
1          0  0.830467     1.000278  3.758047  0.830467  0  0  3.851676   
2          0  0.830467     2.000556  3.756766  0.830467  0  0  3.851531   
3          0  0.830467     3.000834  3.755706  0.830467  0  0  3.851386   
4          0  0.830467     4.001111  3.754780  0.830467  0  0  3.851241   
...      ...       ...          ...       ...       ... .. ..       ...   
47589      0  0.582561  3595.998889  3.578341  0.582561  0  0  3.690012   
47590      0  0.582561  3596.999166  3.578276  0.582561  0  0  3.689996   
47591      0  0.582561  3597.999444  3.578211  0.582561  0  0  3.689979   
47592      0  0.582561  3598.999722  3.578145  0.582561  0  0  3.689962   
47593      0  0.582561  3600.000000  3.578080  0.582561  0  0  3.689946   

            soc       eta  pulse  trajectory  
0      1.000000  0.092106  False           0  
1    